# Task 2: EDA and Sampling Strategy
**Assignment 4 — IEEE CIS Fraud Detection**

This notebook:
1. Explores the IEEE CIS dataset structure and key statistics
2. Documents the sampling strategy (all fraud + 50K non-fraud)
3. Demonstrates the imbalance handling comparison (SMOTE vs class_weight)
4. Shows the chronological split rationale

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')
print('Libraries loaded')

## 1. Load the Working Sample

In [ ]:
from src.data.ingest import load_sample

df = load_sample()
print(f'Shape: {df.shape}')
print(f'Fraud count: {df["isFraud"].sum():,}  ({df["isFraud"].mean():.2%})')
print(f'TransactionDT range: {df["TransactionDT"].min():,} → {df["TransactionDT"].max():,}')
df.head(3)

## 2. Class Imbalance Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution
counts = df['isFraud'].value_counts()
axes[0].bar(['Legitimate (0)', 'Fraud (1)'], counts.values,
            color=['steelblue', 'crimson'])
axes[0].set_title('Class Distribution in Working Sample')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontsize=10)

# TransactionAmt distribution by class
for label, color, name in [(0, 'steelblue', 'Legit'), (1, 'crimson', 'Fraud')]:
    vals = np.log1p(df[df['isFraud'] == label]['TransactionAmt'].dropna())
    axes[1].hist(vals, bins=50, alpha=0.6, color=color, label=name, density=True)
axes[1].set_xlabel('log(1 + TransactionAmt)')
axes[1].set_ylabel('Density')
axes[1].set_title('Transaction Amount Distribution by Class')
axes[1].legend()

plt.tight_layout()
os.makedirs('../outputs/eda', exist_ok=True)
plt.savefig('../outputs/eda/class_distribution.png', dpi=150)
plt.show()
print('Saved → outputs/eda/class_distribution.png')

## 3. Missing Value Analysis

In [ ]:
missing = df.isnull().mean().sort_values(ascending=False)
heavy_missing = missing[missing > 0.30]
print(f'Columns with >30% missing: {len(heavy_missing)}')
print(f'Average missing rate (all cols): {missing.mean():.2%}')

plt.figure(figsize=(14, 4))
top50 = missing.head(50)
colors = ['red' if v > 0.5 else ('orange' if v > 0.30 else 'steelblue') for v in top50]
plt.bar(range(len(top50)), top50.values, color=colors)
plt.axhline(0.30, ls='--', color='orange', label='30% threshold')
plt.xticks(range(len(top50)), top50.index, rotation=90, fontsize=7)
plt.ylabel('Missing Rate')
plt.title('Top 50 Features by Missing Rate')
plt.legend()
plt.tight_layout()
plt.savefig('../outputs/eda/missing_rates.png', dpi=150)
plt.show()
print('Saved → outputs/eda/missing_rates.png')

## 4. Feature Distributions (Key Features)

In [ ]:
# ProductCD distribution by class
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

prod_fraud = df[df['isFraud'] == 1]['ProductCD'].value_counts(normalize=True)
prod_legit = df[df['isFraud'] == 0]['ProductCD'].value_counts(normalize=True)

x = np.arange(len(prod_fraud))
w = 0.35
axes[0].bar(x - w/2, prod_legit.reindex(prod_fraud.index).values, w, label='Legit', color='steelblue')
axes[0].bar(x + w/2, prod_fraud.values, w, label='Fraud', color='crimson')
axes[0].set_xticks(x)
axes[0].set_xticklabels(prod_fraud.index)
axes[0].set_title('ProductCD: Legit vs Fraud')
axes[0].set_ylabel('Proportion')
axes[0].legend()

# M-column coverage
m_cols = [f'M{i}' for i in range(1, 10)]
m_presence = [(col, df[col].notna().mean()) for col in m_cols if col in df.columns]
if m_presence:
    names, vals = zip(*m_presence)
    axes[1].bar(names, vals, color='darkorange')
    axes[1].set_title('M-Column Non-Null Rate')
    axes[1].set_ylabel('Non-null fraction')
    axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('../outputs/eda/feature_overview.png', dpi=150)
plt.show()

## 5. Temporal Distribution (Chronological Split Rationale)

In [ ]:
n = len(df)
val_start  = int(n * 0.70)
test_start = int(n * 0.85)

train_dt = df.iloc[:val_start]['TransactionDT']
val_dt   = df.iloc[val_start:test_start]['TransactionDT']
test_dt  = df.iloc[test_start:]['TransactionDT']

plt.figure(figsize=(12, 3))
plt.plot(train_dt.values, [0] * len(train_dt), '|', color='steelblue',
         alpha=0.3, markersize=5, label=f'Train ({len(train_dt):,})')
plt.plot(val_dt.values,   [1] * len(val_dt),   '|', color='orange',
         alpha=0.3, markersize=5, label=f'Val   ({len(val_dt):,})')
plt.plot(test_dt.values,  [2] * len(test_dt),  '|', color='crimson',
         alpha=0.3, markersize=5, label=f'Test  ({len(test_dt):,})')
plt.xlabel('TransactionDT (seconds)')
plt.yticks([0, 1, 2], ['Train\n(70%)', 'Val\n(15%)', 'Test\n(15%)'])
plt.title('Chronological 70/15/15 Split — No Shuffling')
plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig('../outputs/eda/chronological_split.png', dpi=150)
plt.show()

print('Train DT range:', train_dt.min(), '→', train_dt.max())
print('Val   DT range:', val_dt.min(),   '→', val_dt.max())
print('Test  DT range:', test_dt.min(),  '→', test_dt.max())

## 6. Imbalance Strategy Comparison (SMOTE vs class_weight)

In [ ]:
from src.data.preprocess import preprocess, time_aware_split
from src.data.imbalance import compare_strategies
import os

os.makedirs('../outputs/imbalance', exist_ok=True)
train_raw, val_raw, test_raw = time_aware_split(df)
X_train, y_train, enc, med = preprocess(train_raw, fit=True)
X_test,  y_test,  _,   _   = preprocess(test_raw, encoders=enc, medians=med, fit=False)

results_df = compare_strategies(
    X_train.values, y_train.values,
    X_test.values,  y_test.values,
    out_dir='../outputs/imbalance'
)
results_df

## 7. Key Statistics Summary

In [ ]:
print('=== Working Sample Statistics ===')
print(f'Total rows:          {len(df):,}')
print(f'Fraud rows:          {df["isFraud"].sum():,} ({df["isFraud"].mean():.2%})')
print(f'Non-fraud rows:      {(df["isFraud"]==0).sum():,}')
print(f'Total features:      {df.shape[1]}')
print(f'Numeric features:    {df.select_dtypes(include=[np.number]).shape[1]}')
print(f'Columns >30% null:   {(df.isnull().mean() > 0.30).sum()}')
print(f'TransactionAmt min:  ${df["TransactionAmt"].min():.2f}')
print(f'TransactionAmt max:  ${df["TransactionAmt"].max():.2f}')
print(f'TransactionAmt mean: ${df["TransactionAmt"].mean():.2f}')
print()
print('Fraud by ProductCD:')
print(df.groupby('ProductCD')['isFraud'].agg(['sum','mean','count'])
        .rename(columns={'sum':'fraud_n','mean':'fraud_rate','count':'total'})
        .sort_values('fraud_rate', ascending=False).to_string())